#### Load Dataset

In [24]:
import pandas as pd
import ast

data_path = "../data/usc-x-24-us-election/part_1/"

files = [
    "may_july_chunk_1.csv.gz",
    "may_july_chunk_2.csv.gz",
    "may_july_chunk_3.csv.gz",
    "may_july_chunk_4.csv.gz",
    "may_july_chunk_5.csv.gz"
]

dfs = []
for f in files:
    temp = pd.read_csv(data_path + f, compression='gzip')
    dfs.append(temp)

df = pd.concat(dfs, ignore_index=True)
df.shape

(250000, 32)

In [25]:
df_sample = df[['id', 'text']].dropna().sample(30, random_state=42)
df_sample = df_sample.reset_index(drop=True)
df_sample.shape

(30, 2)

In [26]:
gallup_issues = [
    "The economy",
    "Democracy in the U.S.",
    "Terrorism and national security",
    "Types of Supreme Court justices candidates would pick",
    "Immigration",
    "Education",
    "Healthcare",
    "Gun policy",
    "Abortion",
    "Taxes",
    "Crime",
    "Distribution of income and wealth in the U.S.",
    "The federal budget deficit",
    "Foreign affairs",
    "Situation in Middle East between Israelis and Palestinians",
    "Energy policy",
    "Relations with Russia",
    "Race relations",
    "Relations with China",
    "Trade with other nations",
    "Climate change",
    "Transgender rights"
]
issues_text = "\n".join(gallup_issues)
print(issues_text)

The economy
Democracy in the U.S.
Terrorism and national security
Types of Supreme Court justices candidates would pick
Immigration
Education
Healthcare
Gun policy
Abortion
Taxes
Crime
Distribution of income and wealth in the U.S.
The federal budget deficit
Foreign affairs
Situation in Middle East between Israelis and Palestinians
Energy policy
Relations with Russia
Race relations
Relations with China
Trade with other nations
Climate change
Transgender rights


In [27]:
prompt_template = """Classify the following tweet into ONE of the following issues from the Gallup poll list.

Issues:
{issues}

Tweet:
"{tweet}"

Return ONLY the issue name exactly as written above.
If none match, return "Other".
"""

In [28]:
## test ollama model
import ollama

response = ollama.chat(
    model='qwen3:8b',
    messages=[{'role': 'user', 'content': 'Say hello in one word'}]
)

print(response['message']['content'])

Hello.


In [38]:
import ollama

def classify_tweet(tweet_text):

    print("Processing Tweet...")
    prompt = prompt_template.format(
        issues=issues_text,
        tweet=tweet_text
    )

    response = ollama.generate(
        model='qwen3:8b',
        prompt=prompt,
        options={
            "temperature": 0
        }
    )
    print("MODEL OUTPUT:", response['response'])

    return response['response'].strip()

In [ ]:
### testing on 1 row of df
classify_tweet(df_sample['text'].iloc[0])

Processing Tweet...
MODEL OUTPUT: The economy


'The economy'

In [ ]:
### testing qwen model
import ollama

response = ollama.generate(
    model='qwen3:8b',
    prompt="Classify this text into Economy, Immigration, Crime or Other: Gas prices are too high",
)

print(response['response'])

Economy


In [ ]:
### cleaning tweet
import re

def clean_tweet(text):
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [ ]:
issues = []

for i, tweet in enumerate(df_sample['text']):
    tweet = clean_tweet(tweet_text)
    print(f"-----------------Iteration: {i}-----------------")
    issue = classify_tweet(tweet)
    issues.append(issue)

df_sample['issue'] = issues
df_sample.to_csv("../data/issue_mapping_sample.csv", index=False)

-----------------Iteration: 0-----------------
Processing Tweet...
-----------------Iteration: 1-----------------
Processing Tweet...
-----------------Iteration: 2-----------------
Processing Tweet...
-----------------Iteration: 3-----------------
Processing Tweet...
-----------------Iteration: 4-----------------
Processing Tweet...
-----------------Iteration: 5-----------------
Processing Tweet...
-----------------Iteration: 6-----------------
Processing Tweet...
-----------------Iteration: 7-----------------
Processing Tweet...
-----------------Iteration: 8-----------------
Processing Tweet...
-----------------Iteration: 9-----------------
Processing Tweet...
-----------------Iteration: 10-----------------
Processing Tweet...
-----------------Iteration: 11-----------------
Processing Tweet...
-----------------Iteration: 12-----------------
Processing Tweet...
-----------------Iteration: 13-----------------
Processing Tweet...
-----------------Iteration: 14-----------------
Processing

In [32]:
print(df_sample[['text', 'issue']])

                                                 text issue
0   @kangaroos991 Everything else is in the past. ...      
1   Man who manipulates rates, thus inflation to h...      
2   @Coloradosearch1 @JoJoFromJerz But unlike Dona...      
3   @TrumpSolMAGA @WhaleInsider preparing for a ma...      
4   Biden reportedly puts the blames for Hunter's ...      
5   @_johnnymaga Been saying since his name was an...      
6   @catturd2 @TrumpDailyPosts Tap and read this…....      
7   @CultureWar2020 @LauraLoomer @JoeBiden God in ...      
8   @JDunlap1974 Republicans are going to do nothi...      
9   @BradHolm You vote for Biden, you are voting a...      
10  @ZaleskiLuke @atrupar He is now shopping for a...      
11  @JoeBiden Well, wait a minute with all that mo...      
12  @DiggaveliDoes I’m glad you had Cat or dog pro...      
13  A terrorist operation here on American Soil th...      
14  @realTuckFrumper Florida goes by a Nazi bluepr...      
15  @pdenapo A mi me ha cagado mas veces

In [ ]:
try:
    df_sample.to_csv("../data/sample_issue_classification.csv", index=False)
    print("Saved dataset: ../data/sample_issue_classification.csv")
except Exception as e:
    print("Error while saving:", e)